# Data Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [10, 7]
plt.style.use("seaborn-v0_8")

import seaborn as sns

sns.set(style="darkgrid")

import ipywidgets as widgets
import pandas as pd

## Constants

In [ ]:
PROJECT_ROOT = Path("__file__").resolve().parents[1]

DATA_DPATH = PROJECT_ROOT / "data"
assert DATA_DPATH.exists()

## Data Loading 

In [ ]:
fpath = DATA_DPATH / "src_data" / "train.csv"

df = pd.read_csv(fpath)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["date"] = pd.to_datetime(df["date"])

## Duplicates

In [ ]:
df[df.duplicated()]

## Missing Values

In [ ]:
df.isna().sum()

In [ ]:
df.isna().sum() / len(df)

There are about 4% of missing values in target column

## Date Limits

In [ ]:
df["date"].describe()

## Features

In [ ]:
df["country"].value_counts(dropna=False)

In [ ]:
df["store"].value_counts(dropna=False)

In [ ]:
df["product"].value_counts(dropna=False)

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

plot_data = df["country"].value_counts(dropna=False)
plot_data = plot_data.to_frame(name="cnt").reset_index(names="country")
plot_data["prc"] = plot_data["cnt"] / plot_data["cnt"].sum()
plot_data["labels"] = (plot_data["prc"] * 100).round(1).astype(str) + "%"
plot_data = plot_data.sort_values("country")

sns.barplot(plot_data, x="country", y="cnt", hue="country", ax=axs[0])

for c, label in zip(axs[0].containers, plot_data["labels"], strict=True):
    axs[0].bar_label(c, [label])

axs[0].set_xlabel("Country Name")
axs[0].set_ylabel("Number of Orders")
axs[0].set_title("Country Orders Distribution")


sns.countplot(df, x="country", hue="store", ax=axs[1])

axs[1].set_xlabel("Country Name")
axs[1].set_ylabel("Number of orders")
axs[1].set_title("Contry Orders Distribution by Stores")

plt.tight_layout()
plt.show()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=df["country"].unique()),
    start_date=widgets.DatePicker(value=df["date"].min()),
)
def show_country_data(country: str, start_date):  # noqa: D103
    start_date = pd.Timestamp(start_date)
    plot_df = df[(df["country"] == country) & (df["date"].dt.date >= start_date.date())]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=df["country"].unique()),
    store=widgets.Dropdown(options=df["store"].unique()),
    start_date=widgets.DatePicker(value=df["date"].min()),
)
def show_country_store_data(country: str, store: str, start_date: date):  # noqa: D103
    start_date = pd.Timestamp(start_date)
    plot_df = df[
        (df["country"] == country)
        & (df["store"] == store)
        & (df["date"].dt.date >= start_date.date())
    ]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=df["country"].unique()),
    store=widgets.Dropdown(options=df["store"].unique()),
    products=widgets.Dropdown(options=df["product"].unique()),
    start_date=widgets.DatePicker(value=df["date"].min()),
)
def show_country_store_product_data(country: str, store: str, products: str, start_date: date):  # noqa: D103
    start_date = pd.Timestamp(start_date)
    plot_df = df[
        (df["country"] == country)
        & (df["store"] == store)
        & (df["product"] == products)
        & (df["date"].dt.date >= start_date.date())
    ]

    fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(15, 4))

    axs.plot(plot_df["date"], plot_df["num_sold"])
    axs.set_xlabel("Date")
    axs.set_ylabel("Number of sold stickers")

    plt.show()

## Target

In [ ]:
df["num_sold"].describe()

In [ ]:
@widgets.interact(
    country=widgets.Dropdown(options=df["country"].unique()),
    store=widgets.Dropdown(options=df["store"].unique()),
    products=widgets.Dropdown(options=df["product"].unique()),
)
def show_target_statistics(country: str, store: str, products: str):  # noqa: D103
    plot_df = df[(df["country"] == country) & (df["store"] == store) & (df["product"] == products)]

    na_val = plot_df["num_sold"].isna().sum()
    print(f"Missing values: {na_val} ({na_val / len(plot_df) * 100:.1f}%)")  # noqa: T201

    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15, 4))

    sns.histplot(
        plot_df,
        x="num_sold",
        kde=True,
        bins=30,
        log_scale=False,
        ax=axs[0],
    )
    axs[0].set_xlabel("Number of Sold Stickers")
    axs[0].set_ylabel("Sample Number")
    axs[0].set_title("Number of Sold Stickers Distribution")

    sns.boxplot(plot_df["num_sold"], ax=axs[1])
    axs[1].set_ylabel("Sample Number")
    axs[1].set_title("Sold Sticker Boxplot")

    plt.tight_layout()
    plt.show()